In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Chandni_Chowk_Delhi_IITM_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,NaN,NaN,137.0,179.0,NaN,347.0,NaN,80.0,123.0,NaN,313.0,237.0
1,2,NaN,NaN,NaN,167.0,NaN,329.0,NaN,83.0,89.0,172.0,288.0,200.0
2,3,NaN,NaN,108.0,196.0,286.0,NaN,NaN,89.0,91.0,170.0,302.0,254.0
3,4,NaN,NaN,114.0,NaN,316.0,NaN,NaN,80.0,80.0,204.0,289.0,151.0
4,5,NaN,NaN,120.0,NaN,352.0,341.0,90.0,81.0,78.0,168.0,312.0,151.0
5,6,NaN,NaN,141.0,NaN,307.0,328.0,68.0,78.0,99.0,163.0,299.0,144.0
6,7,NaN,NaN,261.0,223.0,342.0,341.0,NaN,80.0,81.0,142.0,320.0,NaN
7,8,NaN,NaN,184.0,232.0,298.0,353.0,NaN,NaN,96.0,160.0,282.0,309.0
8,9,NaN,NaN,NaN,244.0,204.0,309.0,109.0,NaN,134.0,135.0,229.0,NaN
9,10,NaN,NaN,NaN,255.0,259.0,290.0,158.0,NaN,116.0,106.0,206.0,197.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,15.5,104.923077,137.000000,179.0,253.333333,347.000000,96.2,80.000000,123.000000,173.25,313.000000,237.000000
1,2,15.5,104.923077,185.034483,167.0,253.333333,329.000000,96.2,83.000000,89.000000,172.00,288.000000,200.000000
2,3,15.5,104.923077,108.000000,196.0,286.000000,270.583333,96.2,89.000000,91.000000,170.00,302.000000,254.000000
3,4,15.5,104.923077,114.000000,188.0,316.000000,270.583333,96.2,80.000000,80.000000,204.00,289.000000,151.000000
4,5,15.5,104.923077,120.000000,188.0,352.000000,341.000000,90.0,81.000000,78.000000,168.00,312.000000,151.000000
5,6,15.5,104.923077,141.000000,188.0,307.000000,328.000000,96.2,78.000000,99.000000,163.00,299.000000,144.000000
6,7,15.5,104.923077,261.000000,223.0,342.000000,341.000000,96.2,80.000000,81.000000,142.00,320.000000,191.653846
7,8,15.5,104.923077,184.000000,232.0,298.000000,353.000000,96.2,78.357143,96.000000,160.00,282.000000,191.653846
8,9,15.5,104.923077,185.034483,244.0,204.000000,309.000000,109.0,78.357143,134.000000,135.00,229.000000,191.653846
9,10,15.5,104.923077,185.034483,255.0,259.000000,290.000000,96.2,78.357143,116.000000,106.00,281.151515,197.000000
